# Create penultimate dataframe — Delhi, India

**Pipeline:**
```
land_type_025.csv   (grid + land-cover + Pop + Heat + Buildings)
      ↓  fill_points_land_type_df()
penultimate_df.csv  (adds AQ score, greenspace distance, normalised columns)
```

### Data sources used to build `land_type_025.csv` (pre-processing)
| Feature | Dataset | Resolution | License |
|---------|---------|------------|--------|
| Population | WorldPop India 2024 | 100 m GeoTIFF | CC BY 4.0 |
| Land cover | ESA WorldCover 2021 | 10 m GeoTIFF | CC BY 4.0 |
| Parks / roads / POIs | OpenStreetMap (Geofabrik India) | vector | ODbL |
| Buildings density | Google Open Buildings V3 | polygon | CC BY 4.0 |
| Heat stress | MODIS MOD11A2 (summer mean LST) | ~1 km | NASA open |
| Air quality | CPCB stations (data.gov.in) | station CSV | Govt open |

**Run `Scripts/build_land_type_csv.py` first** to generate `land_type_025.csv`
and `Data/cpcb_stations.csv` before executing this notebook.

In [1]:
import sys
import pathlib
import pandas as pd

ROOT = pathlib.Path().absolute().parent.as_posix()
if ROOT not in sys.path:
    sys.path.append(ROOT)

from helpers import ROOT_FOLDER_PATH, fill_points_land_type_df
print('ROOT_FOLDER_PATH =', ROOT_FOLDER_PATH)

ROOT_FOLDER_PATH = /Users/utkarshsinha/Downloads/Paryavaran


## 1 — Quick sanity check on `land_type_025.csv`

In [2]:
import os
lt_path = ROOT_FOLDER_PATH + '/land_type_025.csv'
assert os.path.exists(lt_path), (
    f'land_type_025.csv not found at {lt_path}.\n'
    'Run Scripts/build_land_type_csv.py first.'
)

lt_df = pd.read_csv(lt_path, index_col=0)
print(f'Grid points: {len(lt_df):,}')
print(lt_df.head(3))
print(lt_df.columns.tolist())

Grid points: 42,299
   Latitude  Longitude  Pop_density  Green_Space  Water  Wetland  Bare_Land  \
0     28.88  76.843218          0.0            0      0        0          0   
1     28.88  76.845786          0.0            0      0        0          0   
2     28.88  76.848353          0.0            0      0        0          0   

   Cropland  Building  Heat_LST  Airport  Railway  Road  Hospital  School  \
0         0         0     305.0        0        0     0         0       0   
1         0         0     305.0        0        0     0         0       0   
2         0         0     305.0        0        0     0         0       0   

   Building_Density  
0               0.0  
1               0.0  
2               0.0  
['Latitude', 'Longitude', 'Pop_density', 'Green_Space', 'Water', 'Wetland', 'Bare_Land', 'Cropland', 'Building', 'Heat_LST', 'Airport', 'Railway', 'Road', 'Hospital', 'School', 'Building_Density']


## 2 — Generate penultimate dataframe (local land_type CSV)

In [3]:
# Runs: dist_nearest_greenspace, CPCB KNN AQ, pop normalisation,
#       heat LST normalisation, building density normalisation
penultimate_df = fill_points_land_type_df()
print(f'\nRows: {len(penultimate_df):,}  Columns: {list(penultimate_df.columns)}')

dist_nearest_greenspace_function complete  [0.0s]
interpolate_aq_for_grid + compute_aq_score complete  [0.01s]
normalise columns complete  [0.0s]

Rows: 42,299  Columns: ['Latitude', 'Longitude', 'Pop_density', 'Green_Space', 'Water', 'Wetland', 'Bare_Land', 'Cropland', 'Building', 'Heat_LST', 'Airport', 'Railway', 'Road', 'Hospital', 'School', 'Building_Density', 'Distance_Nearest_Greenspace', 'PM25', 'PM10', 'NO2', 'AQ_score', 'norm_Pop_density', 'norm_Heat_LST', 'norm_Building_Density']


## 3 — Inspect result

In [4]:
penultimate_df.describe()

,Latitude,Longitude,Pop_density,Green_Space,Water,Wetland,Bare_Land,Cropland,Building,Heat_LST,...,School,Building_Density,Distance_Nearest_Greenspace,PM25,PM10,NO2,AQ_score,norm_Pop_density,norm_Heat_LST,norm_Building_Density
count,42299.000000,42299.000000,42299.0,42299.0,42299.0,42299.0,42299.0,42299.0,42299.0,42299.0,...,42299.0,42299.0,42299.0,42299.000000,42299.000000,42299.000000,42299.000000,42299.0,42299.0,42299.0
mean,28.641073,77.095612,0.0,0.0,0.0,0.0,0.0,0.0,0.0,305.0,...,0.0,0.0,10.0,84.621723,137.721883,36.889588,0.568739,0.0,0.0,0.0
std,0.138213,0.146864,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3.345951,4.799792,1.497631,0.142463,0.0,0.0,0.0
min,28.402425,76.840020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,305.0,...,0.0,0.0,10.0,71.289199,118.418794,31.124970,0.000000,0.0,0.0,0.0
25%,28.521823,76.968397,0.0,0.0,0.0,0.0,0.0,0.0,0.0,305.0,...,0.0,0.0,10.0,82.215106,134.262655,35.824687,0.466222,0.0,0.0,0.0
50%,28.641187,77.095515,0.0,0.0,0.0,0.0,0.0,0.0,0.0,305.0,...,0.0,0.0,10.0,84.996525,138.301946,37.004222,0.585045,0.0,0.0,0.0
75%,28.760683,77.222886,0.0,0.0,0.0,0.0,0.0,0.0,0.0,305.0,...,0.0,0.0,10.0,87.204835,141.597045,37.933926,0.679642,0.0,0.0,0.0
max,28.880000,77.349995,0.0,0.0,0.0,0.0,0.0,0.0,0.0,305.0,...,0.0,0.0,10.0,94.827473,151.762839,41.914881,1.000000,0.0,0.0,0.0


In [5]:
# Check land-type distribution
land_cols = ['Green_Space','Water','Building','Airport','Railway',
             'Hospital','School','Bare_Land','Cropland','Wetland']
existing = [c for c in land_cols if c in penultimate_df.columns]
penultimate_df[existing].sum().sort_values(ascending=False)

Green_Space    0
Water          0
Building       0
Airport        0
Railway        0
Hospital       0
School         0
Bare_Land      0
Cropland       0
Wetland        0
dtype: int64

## 4 — Save to CSV

In [6]:
save_path = ROOT_FOLDER_PATH + '/penultimate_df.csv'
penultimate_df.to_csv(save_path, index=True)
print(f'Saved → {save_path}  ({penultimate_df.memory_usage(deep=True).sum()/1e6:.1f} MB)')

Saved → /Users/utkarshsinha/Downloads/Paryavaran/penultimate_df.csv  (8.5 MB)


## 5 — (Optional) Upload to S3

In [7]:
# from helpers import upload_df_to_s3
# upload_df_to_s3(bucket='your-bucket', df=penultimate_df, key='paryavaran/penultimate_df.csv')